In [1]:
data_path = '/data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract/training/*.bp'
save_path = '/data/xyc/codes/iEEG_network/4_model_BrainPy/3resultAnalysis/Deco_ftract/2trainBPTT_Deco_EndToEnd_ftract-training_20241120.npz'
num_parallel = 50

In [2]:
import os,sys
import brainpy as bp
import brainpy.math as bm
print('bp version: ', bp.__version__)
bm.set_platform('cpu')
bm.set_host_device_count(num_parallel)

import matplotlib.pyplot as plt
import numpy as np
from glob import glob

sys.path.append("/data/xyc/codes/iEEG_network/3_analysis_EC")
sys.path.append("/data/xyc/codes/iEEG_network/4_model_BrainPy")
from func_fetch_conn import fetch_conn_ftract,fetch_conn_enigma,fetch_conn_Kong2021
from func_simulation import def_bodyEC,def_bodyFC,get_parallel_parameters_from_bppaths

bp version:  2.6.0


In [3]:
# loading SC
ctx_labels,sc_ctx, fc_ctx = fetch_conn_enigma() # sc is logged, fc is z-transformed
# ctx_labels,sc_ctx, fc_ctx = fetch_conn_Kong2021() # fc is R-value

# loading EC
# ctx_labels,ftract_ctx,ftract_ctx_amplitud,ftract_ctx_integral,ftract_ctx_duration = fetch_conn_ftract()

# num of nodes
N = len(ctx_labels)

# 除以最大值标准化
struc_conn_matrix = sc_ctx / sc_ctx.max()

In [4]:
bm.set_dt(5e-4) # second; 2000Hz sample rate
bm.set_mode(bm.batching_mode)

Load paths

In [5]:
bppaths = np.asarray(glob(data_path))
print(bppaths.size)

1707


In [6]:
Gs,ws,Is,sigmas,Ss=get_parallel_parameters_from_bppaths(bppaths)

Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract/training/G1.0_w-134.0_I0.0_Adam_lr0.5_wLRx10_epo10k_maxNorm.bp
0/1707
Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract/training/G1.0_w-134.0_I0.0_Adam_lr0.5_wLRx10_epo10k_maxNorm.bp
1/1707
Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract/training/G1.0_w-11.0_I0.0_Adam_lr1.0_wLRx10_epo10k_maxNorm.bp
2/1707
Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract/training/G1.0_w-2.0_I0.0_Adam_lr0.5_wLRx10_epo10k_maxNorm.bp
3/1707
Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract/training/G1.0_w-134.0_I0.0_Adam_lr0.1_wLRx1_epo10k_maxNorm.bp
4/1707
Loading checkpoint from /data/xyc/codes/iEEG_network/4_model_BrainPy/2trainBPTT_Deco_EndToEnd_ftract/training/G1.0_w-146.0_I0.0_Adam_lr1.0_wLRx1_epo1

In [13]:
print(np.isnan(Ss.mean(axis=(1,2))).sum())

0


RUN

In [7]:
bodyEC=def_bodyEC(bm.asarray(struc_conn_matrix))

def bodyEC_parallel(G,w,I,S=0.0):
    _,EC=bodyEC(G,w,I,S)
    return EC

In [ ]:
%%time
ECs = bp.running.jax_parallelize_map(func=bodyEC_parallel,num_parallel=num_parallel,clear_buffer=False,arguments=[Gs,ws,Is,Ss]) 
print(ECs.shape)

(1707, 68, 68)
CPU times: user 44min 31s, sys: 19min 42s, total: 1h 4min 14s
Wall time: 44.2 s


In [9]:
#save
np.savez(save_path, bppaths = bppaths, ECs = ECs,)